# 🤖 ИИ-Агенты на LangChain — Полный курс
По учебному пособию Г. Ю. Звездичева (Москва, 2026)  
**Выполнила:** Манина В. С., группа ББИ2302, МТУСИ

---

## Содержание

| Урок | Тема |
|------|------|
| **Урок 1** | Установка и настройка |
| **Урок 2** | Цепочки, модели, LCEL, stream, batch |
| **Урок 3** | Prompt Templates, Few-shot, Parsers |
| **Урок 4** | Память и история диалога |
| **Урок 5** | Нелинейные пайплайны: Parallel, Branch, Lambda |
| **Урок 6** | Tools — инструменты и вызов функций |
| **Урок 7** | Агенты: ReAct, AgentExecutor |
| **Урок 8** | RAG — поиск по документам |
| **Урок 9** | Продвинутый RAG: Query Rewriting, гибридный поиск |
| **Итог** | ИИ-Агент для нормоконтроля ВКР |

> ⚠️ **Важно:** Запускайте ячейки последовательно. Каждый урок независим после установки пакетов.


---
## 🔧 Урок 1. Установка и настройка окружения


In [ ]:
# Установка пакетов курса (LangChain 0.3: в версии 1.0 AgentExecutor и часть импортов перенесены)
!pip install -q "langchain<1.0" "langchain-openai<0.4" "langchain-community<0.4" "langchain-huggingface<0.4" python-dotenv faiss-cpu pypdf sentence-transformers rank-bm25 beautifulsoup4 pydantic
print('✅ Все пакеты установлены!')


In [ ]:
import os

# ============================================================
# 🔑 ВСТАВЬТЕ СВОЙ КЛЮЧ
# ============================================================

# --- Вариант А: OpenAI ---
# os.environ['OPENAI_API_KEY'] = 'sk-proj-...'
# os.environ['MODEL_NAME'] = 'gpt-4o'

# --- Вариант Б: Rus-GPT (российские карты) ---
from getpass import getpass
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('API-ключ: ')  # ключ не сохраняется в ноутбуке
os.environ['OPENAI_BASE_URL'] = 'https://rus-gpt.com/api/v1'
os.environ['MODEL_NAME'] = 'anthropic/claude-haiku-4.5'

print('✅ Настройки применены')
print(f"Модель: {os.environ.get('MODEL_NAME')}")


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL'),
    temperature=0
)

response = model.invoke("Скажи только: 'Установка прошла успешно!'")
print('Ответ:', response.content)


---
## 📘 Урок 2. Цепочки и модели (LCEL)

**Цель:** Вызывать LLM, строить цепочки, стримить ответы.

### Ключевые концепции
- `ChatPromptTemplate` — шаблон промпта
- LCEL оператор `|` — соединение компонентов
- `.invoke()` / `.stream()` / `.batch()`


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL'),
    temperature=0
)

# 1. Базовый вызов
print('=== 1. Базовый вызов ===')
response = model.invoke('В чём заключается главная трагедия Гамлета?')
print(response.content)


In [ ]:
# 2. Типы сообщений
print('=== 2. Типы сообщений ===')
messages = [
    SystemMessage(content='Ты профессор истории. Отвечай кратко.'),
    HumanMessage(content='Почему распалась Римская империя?'),
]
response = model.invoke(messages)
print(response.content)


In [ ]:
# 3. LCEL цепочка prompt | model
print('=== 3. Цепочка ===')

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты эксперт-литературовед. Анализируй персонажей кратко.'),
    ('human', 'Проанализируй персонажа: {character}'),
])

chain = prompt | model
response = chain.invoke({'character': 'Родион Раскольников'})
print(response.content)


In [ ]:
# 4. Потоковый вывод (stream)
print('=== 4. Stream ===')

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты философ-эссеист. Пиши метафоричные тексты.'),
    ('human', 'Кратко о понятии: {topic}'),
])
chain = prompt | model

for chunk in chain.stream({'topic': 'свобода воли'}):
    print(chunk.content, end='', flush=True)
print()


In [ ]:
# 5. Batch — пакетная обработка
print('=== 5. Batch ===')

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты культуролог. Кратко (1 предложение) опиши главную ценность эпохи.'),
    ('human', 'Эпоха: {era}'),
])
chain = prompt | model

eras = [{'era': 'Античность'}, {'era': 'Возрождение'}, {'era': 'Просвещение'}]
responses = chain.batch(eras)
for era, resp in zip(eras, responses):
    print(f"{era['era']}: {resp.content}")


---
### Самостоятельная работа 1: «Универсальный Секретарь»

Создать программу, пересказывающую новости в разных стилях.
Требования: шаблон `{text}` + `{style}`, потоковый вывод `.stream()`.


In [ ]:
# САМОСТОЯТЕЛЬНАЯ РАБОТА 1 — «Универсальный секретарь»
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL')
)

# 1. Шаблон: роль секретаря + стиль
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты опытный офисный секретарь. Перескажи сообщение коллегам, сохранив все факты, '
               'но строго в стиле: {style}. Ответ — 2–4 предложения.'),
    ('human', '{text}')
])

# 2. Цепочка
chain = prompt | model

input_data = {
    'text': 'Принтер сломался, отчёты задерживаются на час.',
    'style': 'Викторианский джентльмен'
}

print(f"Стиль '{input_data['style']}':")
for chunk in chain.stream(input_data):
    print(chunk.content, end='', flush=True)


---
## 📘 Урок 3. Prompt Templates и парсинг вывода

**Цель:** Few-shot промптинг, структурированный вывод (JSON, Pydantic).

| Инструмент | Результат |
|---|---|
| `StrOutputParser` | str |
| `JsonOutputParser` | dict |
| `PydanticOutputParser` | Pydantic-объект |


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL'),
    temperature=0
)

# StrOutputParser — чистая строка
print('=== StrOutputParser ===')
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Объясняй кратко, в 2-3 предложениях.'),
    ('human', '{question}'),
])
chain = prompt | model | StrOutputParser()
result = chain.invoke({'question': 'Что такое list comprehension?'})
print(type(result), '\n', result)


In [ ]:
# Few-shot промптинг
print('=== Few-shot ===')

examples = [
    {'input': 'Найди баг: def add(a, b): return a - b',
     'output': 'Баг: оператор `-` должен быть `+`. Исправление: return a + b'},
    {'input': 'Найди баг: for i in range(10): print(i) if i == 5: break',
     'output': 'Баг: отсутствует отступ у if. Перенести внутрь цикла.'},
]

example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{input}'),
    ('ai', '{output}'),
])

few_shot = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=examples)

final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты эксперт по отладке Python. Находи баги кратко.'),
    few_shot,
    ('human', '{input}'),
])

chain = final_prompt | model | StrOutputParser()
result = chain.invoke({'input': 'Найди баг: def fact(n): return n * fact(n)'})
print(result)


In [ ]:
from langchain_core.output_parsers import JsonOutputParser

# JsonOutputParser
print('=== JsonOutputParser ===')

parser = JsonOutputParser()
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты эксперт-психолингвист. Отвечай ТОЛЬКО в формате JSON.'),
    ('human', '''Проанализируй текст. Верни JSON с полями:
- political_subtext: скрытые политические взгляды (str)
- hidden_emotions: список эмоций (list)
- manipulation_level: уровень манипуляции 1-10 (int)
- true_intent: истинное намерение (str)

Текст: {text}'''),
])

chain = prompt | model | parser
result = chain.invoke({'text': 'Мы заботимся о безопасности граждан, вводя временные ограничения.'})

print(f"Подтекст: {result['political_subtext']}")
print(f"Эмоции: {result['hidden_emotions']}")
print(f"Манипуляция: {result['manipulation_level']}/10")


In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# PydanticOutputParser — строгая схема
print('=== PydanticOutputParser ===')

class PaintingAnalysis(BaseModel):
    title: str = Field(description='название картины')
    style: str = Field(description='художественный стиль')
    symbols: List[str] = Field(description='список символов на полотне')
    modern_interpretation: str = Field(description='современная интерпретация')

parser = PydanticOutputParser(pydantic_object=PaintingAnalysis)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты эксперт-искусствовед. {format_instructions}'),
    ('human', 'Проанализируй произведение: {painting_description}'),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser
result = chain.invoke({'painting_description': 'Женщина со скрещенными руками на фоне туманного пейзажа'})

print(type(result))
print(f'Название: {result.title}')
print(f'Стиль: {result.style}')
print(f'Символы: {result.symbols}')


In [ ]:
# Агент Научный редактор (бизнес-информатика)
print('=== Научный редактор ===')

examples_sci = [
    {
        'input': 'Мы внедрили новую программу, заказы теперь быстрее.',
        'output': 'Внедрение ИС позволило оптимизировать бизнес-процессы и сократить время обработки заявок.'
    },
    {
        'input': 'Если хранить данные в одном месте, начальству проще смотреть отчёты.',
        'output': 'Централизация данных в рамках DWH повышает прозрачность отчётности и поддержку управленческих решений.'
    }
]

ep = ChatPromptTemplate.from_messages([('human', '{input}'), ('ai', '{output}')])
fs = FewShotChatMessagePromptTemplate(example_prompt=ep, examples=examples_sci)

editor_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты научный редактор в области {discipline}. Используй: KPI, стейкхолдеры, реинжиниринг.'),
    fs,
    ('human', '{text}'),
])

editor = editor_prompt | model | StrOutputParser()
print(editor.invoke({
    'discipline': 'Бизнес-информатика',
    'text': 'Нам надо переделать сайт, чтобы клиенты не уходили из-за тормозов.'
}))


---
## 📘 Урок 4. Память и история диалога

**Ключевые концепции:** `MessagesPlaceholder`, `RunnableWithMessageHistory`, `SQLChatMessageHistory`, session_id


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from langchain_community.chat_message_histories import SQLChatMessageHistory

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL'),
    temperature=0
)

trimmer = trim_messages(
    max_tokens=20,
    strategy='last',
    token_counter=len,
    include_system=True,
    allow_partial=False,
    start_on='human',
)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты ассистент-разработчик. Отвечай кратко и по делу.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}'),
])

chain = (
    RunnablePassthrough.assign(history=lambda x: trimmer.invoke(x['history']))
    | prompt
    | model
)

def get_session_history(session_id: str) -> SQLChatMessageHistory:
    return SQLChatMessageHistory(
        session_id=session_id,
        connection='sqlite:///chat_history.db',
    )

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)

def chat(session_id: str, message: str) -> str:
    return chain_with_history.invoke(
        {'input': message},
        config={'configurable': {'session_id': session_id}},
    ).content

print('=== Изоляция сессий ===')
print('[alice]', chat('alice', 'Меня зовут Алиса, пишу на Python.'))
print('[alice]', chat('alice', 'Какой язык я упомянула?'))
print('[bob]  ', chat('bob', 'Привет! Разрабатываю на Go.'))
print('[bob]  ', chat('bob', 'Что я сказал о своём стеке?'))
print('[alice]', chat('alice', 'Посоветуй библиотеку для HTTP-запросов на моём языке.'))


---
## 📘 Урок 5. Нелинейные пайплайны

| Инструмент | Что делает |
|---|---|
| `RunnableParallel` | Несколько цепочек одновременно |
| `RunnablePassthrough` | Передаёт данные, сохраняя контекст |
| `RunnableLambda` | Python-функция в цепочке |
| `RunnableBranch` | Ветвление по условию |


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL'),
    temperature=0
)

# RunnableParallel — двойной анализ текста
print('=== RunnableParallel ===')

summary_chain = ChatPromptTemplate.from_messages([
    ('system', 'Определи основную мысль текста. Одно предложение.'),
    ('human', '{text}'),
]) | model | StrOutputParser()

review_chain = ChatPromptTemplate.from_messages([
    ('system', 'Найди художественные приёмы. Только список.'),
    ('human', '{text}'),
]) | model | StrOutputParser()

parallel = RunnableParallel(summary=summary_chain, review=review_chain)

result = parallel.invoke({'text': 'Ночь, улица, фонарь, аптека. Бессмысленный и тусклый свет.'})
print('СУТЬ:', result['summary'])
print('ПРИЁМЫ:', result['review'])


In [ ]:
# Накопительный пайплайн с .assign
print('=== Накопительный пайплайн ===')

eval_chain = ChatPromptTemplate.from_messages([
    ('system', 'Оцени сложность. Пиши: Простой/Средний/Сложный и причину.'),
    ('human', '{text}'),
]) | model | StrOutputParser()

summ_chain = ChatPromptTemplate.from_messages([
    ('system', 'Суть текста одним предложением.'),
    ('human', '{text}'),
]) | model | StrOutputParser()

issues_chain = ChatPromptTemplate.from_messages([
    ('system', 'Найди 2 стилистические ошибки. Если нет - напиши: Ошибок нет.'),
    ('human', '{text}'),
]) | model | StrOutputParser()

def build_report(r: dict) -> str:
    return (f'АНАЛИЗ\n{"="*25}\n'
            f'СЛОЖНОСТЬ: {r["complexity"]}\n'
            f'СУТЬ: {r["summary"]}\n'
            f'ЗАМЕЧАНИЯ: {r["issues"]}\n{"="*25}')

pipeline = (
    RunnablePassthrough.assign(complexity=eval_chain)
    | RunnablePassthrough.assign(summary=summ_chain)
    | RunnablePassthrough.assign(issues=issues_chain)
    | RunnableLambda(build_report)
)

text = 'Синергия цифровой трансформации в эпоху постмодернизма приводит к деградации смыслов.'
print(pipeline.invoke({'text': text}))


In [ ]:
# RunnableBranch — маршрутизация
print('=== RunnableBranch ===')

art_chain = ChatPromptTemplate.from_messages([
    ('system', 'Ты эксперт по живописи.'),
    ('human', '{question}'),
]) | model | StrOutputParser()

lit_chain = ChatPromptTemplate.from_messages([
    ('system', 'Ты литературовед.'),
    ('human', '{question}'),
]) | model | StrOutputParser()

culture_chain = ChatPromptTemplate.from_messages([
    ('system', 'Ты гид по мировой культуре.'),
    ('human', '{question}'),
]) | model | StrOutputParser()

def is_art(d): return any(k in d['question'].lower() for k in ['живопись','картина','художник','барокко'])
def is_lit(d): return any(k in d['question'].lower() for k in ['роман','стих','поэт','писатель','метафора'])

branch = RunnableBranch(
    (is_art, art_chain),
    (is_lit, lit_chain),
    culture_chain,
)

print('Искусство:', branch.invoke({'question': 'В чём особенность мазков импрессионистов?'})[:100])
print('Литература:', branch.invoke({'question': 'Главная метафора Преступления и наказания?'})[:100])


---
## 📘 Урок 6. Tools — инструменты и вызов функций

**Схема:** `Вопрос -> Модель -> tool_calls -> Python-функция -> ToolMessage -> Ответ`


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL'),
    temperature=0
)

@tool
def get_weather(city: str) -> str:
    """Возвращает текущую погоду в указанном городе."""
    data = {
        'москва': 'Облачно, +5°C, ветер 12 км/ч',
        'санкт-петербург': 'Дождь, +3°C',
        'лондон': 'Дождь, +8°C',
        'токио': 'Солнечно, +18°C',
    }
    return data.get(city.lower(), f'Нет данных для {city}')

@tool
def calculate(expression: str) -> str:
    """Вычисляет математическое выражение. Пример: '150 * 1.2'"""
    try:
        return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'Ошибка: {e}'

print('Инструмент:', get_weather.name)
print('Описание:', get_weather.description)


In [ ]:
# Полный цикл tool use
tools = [get_weather, calculate]
tools_map = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

def run_with_tools(user_message: str) -> str:
    messages = [HumanMessage(content=user_message)]
    response = model_with_tools.invoke(messages)
    messages.append(response)

    while response.tool_calls:
        for tc in response.tool_calls:
            result = tools_map[tc['name']].invoke(tc['args'])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))
        response = model_with_tools.invoke(messages)
        messages.append(response)

    return response.content

print(run_with_tools('Какая погода в Москве и Лондоне?'))
print(run_with_tools('Сколько будет 1234 * 5678 + 99?'))
print(run_with_tools('Привет, как дела?'))


---
## 📘 Урок 7. Агенты: ReAct, AgentExecutor

**Отличие:** Цепочка = фиксированный сценарий. Агент = сам решает что делать.


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor, create_tool_calling_agent

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL'),
    temperature=0
)

@tool
def get_weather(city: str) -> str:
    """Возвращает текущую погоду в указанном городе."""
    data = {'москва': 'Облачно, +5°C', 'лондон': 'Дождь, +8°C', 'токио': 'Солнечно, +18°C'}
    return data.get(city.lower(), 'Нет данных')

@tool
def calculate(expression: str) -> str:
    """Вычисляет математическое выражение. Пример: '(12 + 5) * 3'"""
    try: return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e: return f'Ошибка: {e}'

@tool
def search_wiki(query: str) -> str:
    """Ищет краткую информацию по запросу (Wikipedia)."""
    db = {
        'langchain': 'LangChain - фреймворк для создания приложений с LLM.',
        'rag': 'RAG - метод обогащения LLM знаниями из внешней базы.',
        'python': 'Python - высокоуровневый язык программирования.',
    }
    for k, v in db.items():
        if k in query.lower(): return v
    return f'Информация по запросу {query!r} не найдена.'

tools = [get_weather, calculate, search_wiki]

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Ты полезный ассистент. Используй инструменты для актуальных данных.'),
    MessagesPlaceholder(variable_name='chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad'),
])

agent = create_tool_calling_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)

result = agent_executor.invoke({'input': 'Какая погода в Москве? И сколько будет 15 * 7?'})
print('Ответ:', result['output'])


In [ ]:
# Агент с памятью диалога
print('=== Агент с памятью ===')
chat_history = []

def chat_with_agent(message: str) -> str:
    result = agent_executor.invoke({'input': message, 'chat_history': chat_history})
    chat_history.append(HumanMessage(content=message))
    chat_history.append(AIMessage(content=result['output']))
    return result['output']

print(chat_with_agent('Что такое LangChain?'))
print('---')
print(chat_with_agent('А что такое RAG? Это связано с тем, что я спрашивал?'))


---
## 📘 Урок 8. RAG — поиск по документам

```
Документы -> Splitter -> Embeddings -> FAISS
Вопрос -> Retriever -> Чанки -> Промпт + LLM -> Ответ
```


In [ ]:
!pip install faiss-cpu langchain-huggingface sentence-transformers -q
print('✅ RAG пакеты готовы')


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL'),
    temperature=0
)

# База знаний
documents = [
    Document(page_content=(
        'LangChain - фреймворк для разработки приложений на основе LLM. '
        'Поддерживает цепочки, агентов, память и RAG. '
        'LCEL (LangChain Expression Language) строит пайплайны через оператор |.'
    ), metadata={'source': 'langchain.txt'}),
    Document(page_content=(
        'RAG (Retrieval-Augmented Generation) - обогащение LLM внешними знаниями. '
        'Компоненты: Document Loader, Text Splitter, Embeddings, VectorStore, Retriever. '
        'FAISS - библиотека Facebook для эффективного векторного поиска.'
    ), metadata={'source': 'rag.txt'}),
    Document(page_content=(
        'Агенты работают по паттерну ReAct: Reasoning + Acting. '
        'AgentExecutor управляет циклом агента. '
        'Отличие от цепочки: агент адаптивен, сам решает последовательность шагов.'
    ), metadata={'source': 'agents.txt'}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f'Подготовлено {len(chunks)} чанков')


In [ ]:
# Эмбеддинги и FAISS
print('Создаём эмбеддинги (первый запуск скачивает модель ~90MB)...')

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print('FAISS-индекс создан')

# Проверка поиска
results = retriever.invoke('Что такое RAG?')
print(f'Найдено {len(results)} чанков:')
for i, d in enumerate(results, 1):
    print(f'  [{i}] {d.page_content[:80]}...')


In [ ]:
# Полная RAG-цепочка
def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'Ты помощник. Отвечай ТОЛЬКО на основе контекста.\n'
        'Если ответа нет - скажи честно.\n\nКонтекст:\n{context}'
    )),
    ('human', '{question}'),
])

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

questions = ['Что такое LangChain?', 'Как работает RAG?', 'Чем агент отличается от цепочки?']
for q in questions:
    print(f'Q: {q}')
    print(f'A: {rag_chain.invoke(q)}\n')


---
## 📘 Урок 9. Продвинутый RAG: Query Rewriting и гибридный поиск

1. **Query Rewriting** - LLM переформулирует вопрос с учётом контекста диалога
2. **BM25** - лексический поиск по точным терминам
3. **EnsembleRetriever** - слияние FAISS + BM25 через Reciprocal Rank Fusion


In [ ]:
!pip install rank-bm25 -q
print('✅ BM25 готов')


In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

model = ChatOpenAI(
    model=os.environ.get('MODEL_NAME', 'gpt-4o'),
    openai_api_key=os.environ['OPENAI_API_KEY'],
    base_url=os.environ.get('OPENAI_BASE_URL'),
    temperature=0
)

raw_docs = [
    Document(page_content='LangChain - фреймворк для создания приложений с LLM. Поддерживает цепочки, агентов, RAG.'),
    Document(page_content='RAG (Retrieval-Augmented Generation) - метод обогащения LLM внешними знаниями из базы.'),
    Document(page_content='FAISS - библиотека Facebook для векторного поиска. Применяется в RAG-системах.'),
    Document(page_content='BM25 - алгоритм лексического поиска. Эффективен для поиска точных терминов типа ГОСТ.'),
    Document(page_content='EnsembleRetriever объединяет FAISS и BM25 методом RRF (Reciprocal Rank Fusion).'),
    Document(page_content='Query Rewriting - переформулировка запроса пользователя с учётом истории диалога.'),
    Document(page_content='Model Context Protocol (MCP) - стандарт подключения инструментов к LLM-агентам.'),
    Document(page_content='ReAct (Reasoning + Acting) - паттерн агентов: рассуждение -> действие -> наблюдение.'),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
docs = splitter.split_documents(raw_docs)

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)
faiss_store = FAISS.from_documents(docs, embeddings)
faiss_retr = faiss_store.as_retriever(search_kwargs={'k': 3})

bm25_retr = BM25Retriever.from_documents(docs)
bm25_retr.k = 3

hybrid = EnsembleRetriever(retrievers=[faiss_retr, bm25_retr], weights=[0.5, 0.5])
print('Гибридный ретривер готов')


In [ ]:
# Query Rewriting + память
rewriter = ChatPromptTemplate.from_messages([
    ('system', 'Переформулируй вопрос пользователя в самодостаточный поисковый запрос. Только запрос, без пояснений.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', 'Вопрос: {question}\nПоисковый запрос:'),
]) | model | StrOutputParser()

def log_query(q: str) -> str:
    print(f'[Query Rewriting] -> {q!r}')
    return q

format_docs = lambda docs: '\n\n'.join(d.page_content for d in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Отвечай ТОЛЬКО на основе контекста. Контекст: {context}'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}'),
])

adv_rag = (
    {
        'context': rewriter | RunnableLambda(log_query) | hybrid | format_docs,
        'question': lambda x: x['question'],
        'history': lambda x: x['history'],
    }
    | rag_prompt | model | StrOutputParser()
)

store = {}
def get_hist(sid): 
    if sid not in store: store[sid] = ChatMessageHistory()
    return store[sid]

rag_with_mem = RunnableWithMessageHistory(
    adv_rag, get_hist, input_messages_key='question', history_messages_key='history'
)

def ask(q, s='demo'):
    return rag_with_mem.invoke({'question': q}, config={'configurable': {'session_id': s}})

print('Вы: Что такое Model Context Protocol?')
print('Агент:', ask('Что такое Model Context Protocol?'))
print('\nВы: А зачем он нужен?')
print('Агент:', ask('А зачем он нужен?'))


---
## 🏆 Итоговая работа: ИИ-Агент для нормоконтроля ВКР

**Цель:** Агент проверяет дипломные работы по нормативным документам.

### Обязательные модули
1. **Индексирование** - PDF -> чанки -> FAISS (с кэшированием)
2. **Гибридный поиск** - FAISS + BM25 + EnsembleRetriever
3. **Query Rewriting** - переформулировка с логированием
4. **Диалог с памятью** - ChatMessageHistory + /exit

### Тест-кейсы
1. Шрифт и интервал основной части
2. Список литературы -> уточнение об электронном ресурсе
3. Проверка фрагмента введения
4. Устойчивость к галлюцинациям (вопрос о Гарварде)


In [ ]:
import os
from pathlib import Path
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# --- Конфигурация ---
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'ВСТАВЬТЕ_КЛЮЧ')
OPENAI_BASE_URL = os.environ.get('OPENAI_BASE_URL')
MODEL_NAME = os.environ.get('MODEL_NAME', 'gpt-4o')
INDEX_PATH = 'vcr_faiss_index'

model = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    temperature=0
)

# --- Нормативная база ВКР ---
norm_docs = [
    Document(page_content=(
        'ГОСТ 7.0.11-2011. Основной шрифт: Times New Roman, 14 пт. '
        'Межстрочный интервал: полуторный (1.5). '
        'Поля: левое 30мм, правое 15мм, верхнее и нижнее 20мм. '
        'Нумерация страниц: арабские цифры, внизу по центру.'
    ), metadata={'source': 'ГОСТ.pdf'}),
    Document(page_content=(
        'ГОСТ 7.0.11-2011. Заголовки разделов: заглавными буквами, по центру, жирный. '
        'Заголовки подразделов: строчными буквами, по центру, жирный. '
        'После заголовка точка не ставится. Каждый раздел начинается с новой страницы.'
    ), metadata={'source': 'ГОСТ.pdf'}),
    Document(page_content=(
        'ГОСТ Р 7.0.5-2008. Список литературы в алфавитном порядке. '
        'Книга: Автор. Название. - Город: Издательство, Год. - Страниц с. '
        'Электронный ресурс: Автор. Название [Электронный ресурс]. URL: адрес (дата обращения: ДД.ММ.ГГГГ).'
    ), metadata={'source': 'ГОСТ.pdf'}),
    Document(page_content=(
        'Методические указания. Структура ВКР: Титульный лист, Содержание, Введение, '
        'Основная часть, Заключение, Список источников, Приложения. '
        'Объём: 60-80 страниц. Таблицы и рисунки нумеруются и имеют названия.'
    ), metadata={'source': 'Метод.Указания.pdf'}),
    Document(page_content=(
        'Приказ ректора N42. Введение: 3-5 страниц. '
        'Актуальность: не менее 0.5 страницы. '
        'Цель исследования: одна конкретная цель. '
        'Задачи исследования: 4-6 задач, соответствующих разделам. '
        'Цели и задачи должны быть взаимосвязаны логически.'
    ), metadata={'source': 'Приказ.pdf'}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
chunks = splitter.split_documents(norm_docs)
print(f'Подготовлено {len(chunks)} чанков')

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)

# Кэширование индекса
if Path(INDEX_PATH).exists():
    print('Загружаем FAISS из кэша...')
    faiss_store = FAISS.load_local(INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
else:
    print('Создаём FAISS-индекс...')
    faiss_store = FAISS.from_documents(chunks, embeddings)
    faiss_store.save_local(INDEX_PATH)
    print(f'Индекс сохранён в {INDEX_PATH!r}')

faiss_retr = faiss_store.as_retriever(search_kwargs={'k': 4})
bm25_retr = BM25Retriever.from_documents(chunks)
bm25_retr.k = 4
hybrid = EnsembleRetriever(retrievers=[faiss_retr, bm25_retr], weights=[0.5, 0.5])
print('Гибридный ретривер готов')


In [ ]:
# Query Rewriting + Диалог с памятью
rewriter = ChatPromptTemplate.from_messages([
    ('system', 'Переформулируй вопрос студента в самодостаточный поисковый запрос для нормативной базы. Только запрос.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', 'Вопрос: {question}\nПоисковый запрос:'),
]) | model | StrOutputParser()

def log_q(q):
    print(f'[Query Rewriting] -> {q!r}')
    return q

fmt = lambda docs: '\n\n---\n\n'.join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)

norm_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'Ты ИИ-Агент нормоконтроля ВКР.\n'
        'Правила:\n'
        '1. Отвечай ТОЛЬКО по контексту из нормативных документов.\n'
        '2. Если информации нет - скажи: Данная информация отсутствует в регламентах.\n'
        '3. Всегда указывай источник (ГОСТ.pdf / Приказ.pdf / Метод.Указания.pdf).\n'
        '4. При проверке фрагмента - давай структурированную рецензию.\n\n'
        'Контекст:\n{context}'
    )),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}'),
])

vcr_chain = (
    {
        'context': rewriter | RunnableLambda(log_q) | hybrid | fmt,
        'question': lambda x: x['question'],
        'history': lambda x: x['history'],
    }
    | norm_prompt | model | StrOutputParser()
)

sessions = {}
def get_hist(sid):
    if sid not in sessions: sessions[sid] = ChatMessageHistory()
    return sessions[sid]

vcr_agent = RunnableWithMessageHistory(
    vcr_chain, get_hist, input_messages_key='question', history_messages_key='history'
)

def ask(q, s='student'):
    return vcr_agent.invoke({'question': q}, config={'configurable': {'session_id': s}})

print('ИИ-Агент нормоконтроля ВКР готов!')


In [ ]:
# Тест-кейсы
print('='*60)
print('ТЕСТ 1: Шрифт и интервал')
print('='*60)
print(ask('Какой шрифт и интервал в основной части диплома?', 't1'))

print('\n' + '='*60)
print('ТЕСТ 2: Контекстуальный диалог (Query Rewriting)')
print('='*60)
print(ask('Как оформлять список литературы по ГОСТу?', 't2'))
print('\n---')
print(ask('А если это электронный ресурс?', 't2'))

print('\n' + '='*60)
print('ТЕСТ 3: Проверка фрагмента ВКР')
print('='*60)
print(ask('Проверь: "Целью данной работы является исследование MCP..." Напиши замечания.', 't3'))

print('\n' + '='*60)
print('ТЕСТ 4: Устойчивость к галлюцинациям')
print('='*60)
print(ask('Каковы правила оформления дипломов в Гарвардском университете?', 't4'))


In [ ]:
# Интерактивный чат-режим
# Введите /exit для выхода
print('ИИ-Агент нормоконтроля ВКР')
print('Документы: ГОСТ.pdf | Приказ.pdf | Метод.Указания.pdf')
print('Введите /exit для выхода\n')

while True:
    try:
        user_input = input('Вы: ').strip()
    except EOFError:
        break
    if not user_input: continue
    if user_input.lower() == '/exit':
        print('До свидания! Удачи с защитой! 🎓')
        break
    print(f'\nАгент: {ask(user_input, "interactive")}\n')


---
## 📊 Итоги курса

| Урок | Ключевой инструмент |
|------|--------------------|
| 2 | LCEL: `prompt | model | parser` |
| 3 | `PydanticOutputParser`, `FewShot` |
| 4 | `RunnableWithMessageHistory`, SQLite |
| 5 | `RunnableParallel`, `RunnableBranch` |
| 6 | `@tool`, `.bind_tools()`, `ToolMessage` |
| 7 | `AgentExecutor`, ReAct |
| 8 | FAISS, `HuggingFaceEmbeddings` |
| 9 | BM25, `EnsembleRetriever`, Query Rewriting |
| Итог | Все технологии |

---
### Что дальше
- **Re-ranking**: cross-encoder для переранжирования чанков
- **LangGraph**: циклические агенты с checkpoints
- **FastAPI**: агент как API-сервис
- **LangSmith**: мониторинг и трейсинг
